## Module imports with loading the dataset:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import matplotlib.ticker as ticker

df = pd.read_excel('../data/retail_dataset_cleaned.xlsx')

## Sales Calculations:

In [ ]:
total_revenue = df['total_sales'].sum()
avg_order_value = df.groupby('retailer_id')['total_sales'].sum().mean()
total_products_sold = df['units_sold'].sum()

print(f"Total Revenue: Rs. {total_revenue:,.2f},")
print(f"Average Order Value: Rs. {avg_order_value:,.2f},")
print(f"Total Products Sold: {total_products_sold:}")

## Revenue Over Time

In [ ]:
df['invoice_date'] = pd.to_datetime(df['invoice_date'])

df['Year'] = df['invoice_date'].dt.year
df['Month'] = df['invoice_date'].dt.month
df['Quarter'] = df['invoice_date'].dt.quarter
df['Day_of_Week'] = df['invoice_date'].dt.day_name()
df['Month_Name'] = df['invoice_date'].dt.month_name()

# Monthly
monthly_revenue = df.groupby(['Year', 'Month'])['total_sales'].sum().reset_index() # reset_index() to convert groupby object to DataFrame
monthly_revenue['Year_Month'] = monthly_revenue['Year'].astype(str) + '-' + monthly_revenue['Month'].astype(str).str.zfill(2)

plt.figure(figsize=(14, 6))
plt.plot(monthly_revenue['Year_Month'], monthly_revenue['total_sales'], marker='o')
plt.title('Monthly Revenue Trend', fontsize=16, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue (Rs.)')
plt.xticks(rotation=45)
plt.grid(alpha=0.3)
plt.tight_layout()

# Change y-axis to show in millions with commas
ax = plt.gca() # Gets the current active axis layout
ax.get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.show()

In [ ]:
# Yearly
yearly_revenue = df.groupby('Year')['total_sales'].sum()
print("\nYearly Revenue:")
print(yearly_revenue)
print("\nYear-over-Year Growth (from 2024 to 2025):")

# yoy_growth = (yearly_revenue.pct_change() * 100).fillna(0)
yoy_growth = (yearly_revenue.pct_change() * 100).map('{:.2f}%'.format).replace('nan%', 'N/A')

# any of the above two lines can be used to calculate yoy_growth, the second one formats the output as percentage and replaces NaN with 'N/A'

print(yoy_growth)

## Seasonal Analysis:

In [ ]:
quarterly_sales = df.groupby(['Year', 'Quarter'])['total_sales'].sum().reset_index()
print("\nQuarterly Sales:")
print(quarterly_sales)

monthly_sales = df.groupby('Month_Name')['total_sales'].sum().sort_values(ascending=False).head(5)
print("\nBest Performing Months:")
print(monthly_sales)

dow_sales = df.groupby('Day_of_Week')['total_sales'].sum().sort_values(ascending=False).head(3)
print("\nSales by Day of Week (best 3):")
print(dow_sales)

In [ ]:
plt.figure(figsize=(14, 6))

month_order = ['January', 'February', 'March', 'April', 'May', 'June', 
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_sales_ordered = df.groupby('Month_Name')['total_sales'].sum().reindex(month_order)
plt.bar(range(12), monthly_sales_ordered.values)
plt.xlabel('Sales by Month')
plt.ylabel('Revenue (Rs.)')

ax = plt.gca()
ax.get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

quarterly_total = df.groupby('Quarter')['total_sales'].sum()
plt.bar(quarterly_total.index, quarterly_total.values, color='orange')
plt.title('Sales every Quarter-Year')
plt.xlabel('Quarter')
plt.ylabel('Revenue (Rs.)')

ax = plt.gca()
ax.get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_sales_ordered = df.groupby('Day_of_Week')['total_sales'].sum().reindex(day_order)
plt.bar(range(7), dow_sales_ordered.values, color='green')
plt.xticks(range(7), [d[:3] for d in day_order], rotation=45)
plt.title('Sales by Day of Week')
plt.ylabel('Revenue (Rs.)')

ax = plt.gca()
ax.get_yaxis().set_major_formatter(ticker.FuncFormatter(lambda x, p: format(int(x), ',')))
plt.show()